In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from robusta_hmf import Robusta
import time
import itertools

In [ ]:
start = time.perf_counter()

In [ ]:
#define constants
c = 299792.458 #km/s
LN10 = np.log(10.)
MAX_IVAR = 2.5e3
MIN_IVAR = 0.1

In [ ]:
#read in the data and deal with radial velocities
with open('hot_stars_4500_2026-06-30_data.pkl','rb') as f:
     fluxes, loglam, ivars, continuua, spec_files, rvs = pickle.load(f)
print(fluxes.shape, loglam.shape, ivars.shape, continuua.shape, spec_files.shape)

In [ ]:
#log lambda values have been corrupted
print(10**loglam)
print(loglam[1:] - loglam[:-1])
print(np.median(loglam[1:] - loglam[:-1]))
delta_log_lambda_pixel = np.median(loglam[1:] - loglam[:-1])

In [ ]:
#get ready to do pixel shifts
delta_log_lambdas = (rvs / c) / LN10
delta_pixels = np.round(delta_log_lambdas/delta_log_lambda_pixel).astype(int)

print(delta_log_lambdas, delta_pixels)

In [ ]:
#shift spectra to the rest frame
## johanna is not going to like this

rest_fluxes = np.zeros_like(fluxes) + 1.
rest_ivars = np.zeros_like(ivars)
rest_continuua = np.zeros_like(continuua) + np.nan
for i, dp in enumerate(delta_pixels):
    if dp < 0:
        rest_fluxes[i, -dp:] = fluxes[i, :dp]
        rest_ivars[i, -dp:] = ivars[i, :dp]
        rest_continuua[i, -dp:] = continuua[i, :dp]
        
    elif dp > 0:
        rest_fluxes[i, :-dp] = fluxes[i, dp:]
        rest_ivars[i, :-dp] = ivars[i, dp:]
        rest_continuua[i, :-dp] = continuua[i, dp:]
        
    else:
        rest_fluxes[i, :] = fluxes[i, :]
        rest_ivars[i, :] = ivars[i, :]
        rest_continuua[i, :] = continuua[i, :]


In [ ]:
# spot check the above
print(np.min(delta_pixels))
largest_shift = np.argmin(delta_pixels)
print(largest_shift)

plt.plot(10**loglam, fluxes[largest_shift], 'k-')
plt.plot(10**loglam, rest_fluxes[largest_shift], 'r-')
plt.xlim(4845, 4870)
plt.axvline(4471.48)
plt.ylim(0.3,1.05)
plt.show()


In [ ]:
#fix nans and infinities
bad = np.logical_not(np.isfinite(rest_fluxes))
rest_fluxes[bad] = 1
rest_ivars[bad] = 0

bad = np.logical_not(np.isfinite(rest_ivars))
rest_fluxes[bad] = 1
rest_ivars[bad] = 0

bad = np.logical_or((rest_fluxes > 2.0), (rest_fluxes < 0))
rest_fluxes[bad] = 1
rest_ivars[bad] = 0

bad = rest_ivars > MAX_IVAR
# rest_fluxes[bad] = 1
# rest_ivars[bad] = 0
rest_ivars[bad] = MAX_IVAR

bad = ivars < MIN_IVAR
rest_fluxes[bad] = 1.0
rest_ivars[bad] = MIN_IVAR

In [ ]:
print(f"flux median: {np.median(rest_fluxes)}, ivar median{np.median(rest_ivars)}")
print(np.max(rest_fluxes), np.min(rest_fluxes))
print(np.max(rest_ivars), np.min(rest_ivars))

In [ ]:
#we are using the rest frame data as our data 
data = rest_fluxes
weights = rest_ivars

In [ ]:
# split data into an A sample and a B sample
N, M = fluxes.shape
foo = np.arange(N)
rng = np.random.default_rng(17)
rng.shuffle(foo)
Aindx = foo[:N // 2]
Bindx = foo[N // 2:]


In [ ]:
# plot some of the data from sample B just to look
f = plt.figure(figsize=(15, 15))
offset = 0.5
for i in range(16):
    f.gca().plot(data[Bindx[i]] + i * offset, color="k")

In [ ]:
# train two Robusta models
K = 16
#foo, bar = 2500, 4500
modelA = Robusta(rank=K, robust=True, robust_scale = 1, robust_nu = 3)
modelB = Robusta(rank=K, robust=True)
fooA = modelA.fit(data[Aindx], weights[Aindx], max_iter = 10000)
#fooA = modelA.fit(data[Aindx, foo:bar], weights[Aindx, foo:bar], max_iter = 10000)
#fooB = modelB.fit(data[Bindx, foo:bar], weights[Bindx, foo:bar], max_iter = 10000)


In [ ]:
K_GRID = [8, 16, 24]
NU_GRID = [1, 3, 10]
SCALE_GRID = [1, 2, 4]
SEED = 17
MAX_ITER = 10000

results = []

for K, nu, scale in itertools.product(K_GRID, NU_GRID, SCALE_GRID):

    modelA = Robusta(rank=K, robust=True, robust_nu=nu, robust_scale=scale)

    # fit on A
    modelA.fit(data[Aindx], weights[Aindx], max_iter=MAX_ITER, seed=SEED)

    # infer coefficients for B, holding A's G fixed, then synthesize
    state, _ = modelA.infer(data[Bindx], weights[Bindx])
    synth_B = modelA.synthesize(state)

    # MAD statistic: median absolute difference between observed and synthesized B
    mad = np.median(np.abs(data[Bindx] - synth_B))

    results.append({"K": K, "nu": nu, "scale": scale, "MAD": mad})
    print(f"K={K:2d}  nu={nu:2d}  scale={scale}  ->  MAD={mad:.5f}")

df = pd.DataFrame(results)

In [ ]:
table = df.pivot_table(index="K", columns=["nu", "scale"], values="MAD")

In [ ]:
table